# cell2location — Step 1: estimate reference cell-type signatures

This notebook is generated from the cell2location `spatial-mapping` skill.
See [../SKILL.md](../SKILL.md) for the full workflow.

It runs `cell2location.models.RegressionModel` on an scRNA-seq reference AnnData
to produce a `signatures.csv` (genes x cell_types, batch-corrected, linear scale)
consumed by step 2 ([step2_spatial_mapping.ipynb](step2_spatial_mapping.ipynb)).

Inputs are configured via the papermill parameters cell below. See
[../reference/hyperparameters_extract.md §2](../reference/hyperparameters_extract.md)
for the gene-filter and signature-estimation rules.


In [ ]:
# === PARAMETERS (papermill) ===
ref_h5ad_path = ""                              # path to scRNA reference AnnData
batch_key = "sample"                            # column for batch (drives detection_mean_y_e + s_g_gene_add)
labels_key = "cell_type"                        # column for cell-type labels (drives per_cluster_mu_fg)
categorical_covariate_keys = []                 # extra categorical covariates (e.g. ['10x_kit', 'donor'])
                                                # drive per-gene tech regression detection_tech_gene_tg.
                                                # Use this for multi-technology references.
continuous_covariate_keys = []                  # rare
gene_filter_cell_count_cutoff = 15
gene_filter_cell_percentage_cutoff2 = 0.03
gene_filter_nonz_mean_cutoff = 1.12
max_epochs = None                               # None -> auto: min(round(20000/n_cells * 400), 400)
output_dir = "./signatures_output"
output_name = "ref_signatures"


In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata
import cell2location
from cell2location.utils.filtering import filter_genes
from cell2location.models import RegressionModel


## Load reference data

In [ ]:
adata_ref = sc.read_h5ad(ref_h5ad_path)
print(f"Loaded reference: {adata_ref.n_obs} cells, {adata_ref.n_vars} genes")
print(f"labels_key='{labels_key}' has {adata_ref.obs[labels_key].nunique()} unique values")


## Gene filtering (supplement Fig S1 rule)

In [ ]:
# Two-cutoff gene selection (Fig S1):
#   1. genes with count > 0 in >5% of cells, OR
#   2. genes with count > 0 in 5% > cells > 10 cells AND mean expression > 1.12
selected = filter_genes(
    adata_ref,
    cell_count_cutoff=gene_filter_cell_count_cutoff,
    cell_percentage_cutoff2=gene_filter_cell_percentage_cutoff2,
    nonz_mean_cutoff=gene_filter_nonz_mean_cutoff,
)
adata_ref = adata_ref[:, selected].copy()
print(f"After filtering: {adata_ref.n_vars} genes")


## Register data + train RegressionModel

In [ ]:
# Register data with cell2location's reference model.
# - batch_key drives per-batch detection (detection_mean_y_e) and additive
#   background (s_g_gene_add).
# - categorical_covariate_keys drive per-gene technology regression
#   (detection_tech_gene_tg). Use this for multi-technology references
#   (e.g. 10x v2 + v3 + Smart-seq); do not lump tech into batch_key.
RegressionModel.setup_anndata(
    adata=adata_ref,
    batch_key=batch_key,
    labels_key=labels_key,
    categorical_covariate_keys=categorical_covariate_keys if categorical_covariate_keys else None,
    continuous_covariate_keys=continuous_covariate_keys if continuous_covariate_keys else None,
)


In [ ]:
# Auto max_epochs if not set: min(round(20000/n_cells * 400), 400)
if max_epochs is None:
    max_epochs = min(round(20000 / adata_ref.n_obs * 400), 400)
print(f"Training for {max_epochs} epochs")

mod_ref = RegressionModel(adata_ref)
mod_ref.view_anndata_setup()
mod_ref.train(max_epochs=max_epochs, accelerator='gpu', train_size=1, batch_size=2500)


## Export posterior + save signatures

In [ ]:
# Export posterior
adata_ref = mod_ref.export_posterior(
    adata_ref,
    sample_kwargs={'num_samples': 1000, 'batch_size': 2500, 'accelerator': 'gpu'},
)

# QC: ELBO history
mod_ref.plot_history(20)


In [ ]:
# Save model + signatures CSV
run_dir = os.path.join(output_dir, output_name)
os.makedirs(run_dir, exist_ok=True)
mod_ref.save(run_dir, overwrite=True)
adata_ref.write(os.path.join(run_dir, "ref_signatures.h5ad"))

# Extract per-cell-type signature means (linear scale, batch-corrected)
# inf_aver is the input to step2's Cell2location model.
if 'means_per_cluster_mu_fg' in adata_ref.varm.keys():
    inf_aver = adata_ref.varm['means_per_cluster_mu_fg'][[
        f'means_per_cluster_mu_fg_{i}' for i in adata_ref.uns['mod']['factor_names']
    ]].copy()
else:
    inf_aver = adata_ref.var[[
        f'means_per_cluster_mu_fg_{i}' for i in adata_ref.uns['mod']['factor_names']
    ]].copy()
inf_aver.columns = adata_ref.uns['mod']['factor_names']

signatures_csv = os.path.join(run_dir, "signatures.csv")
inf_aver.to_csv(signatures_csv)
print(f"Wrote signatures to {signatures_csv}: {inf_aver.shape}")
